# GraphRAG Quickstart

This notebook demonstrates the basic workflow:
1. Ingest documents
2. Build knowledge graph
3. Query with GraphRAG


In [1]:
# Setup: Add project root to Python path for imports
import sys
from pathlib import Path

# Find project root by looking for pyproject.toml or graphrag package
current = Path.cwd().resolve()

# Try going up from current directory
project_root = None
for parent in [current] + list(current.parents):
    if (parent / "pyproject.toml").exists() or (parent / "graphrag").exists():
        project_root = parent
        break

# If not found, try relative to this notebook file location
if project_root is None:
    # When running in Jupyter, __file__ might not exist, so we use a fallback
    try:
        notebook_file = Path(__file__).parent.parent.parent
        if (notebook_file / "pyproject.toml").exists():
            project_root = notebook_file
    except:
        # Fallback: assume we're 2 levels down from project root
        project_root = Path.cwd().parent.parent

# Add project root to Python path
if project_root and str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"Project root: {project_root}")
print(f"Python path configured: {str(project_root) in sys.path if project_root else 'Failed to find project root'}")


Project root: C:\Users\rishi\OneDrive\College\CS598\graphRAG
Python path configured: True


In [2]:
# Import required modules
import asyncio
import glob
from pathlib import Path
from graphrag.app.services.pipeline import pipeline
from graphrag.app.api.routers import query
from graphrag.app.api.schemas import QueryRequest
from graphrag.tests.pubmed_test_data import create_pubmed_test_documents, get_sample_medical_queries

print("✓ Imports successful")


2025-11-01 11:09:32 [info     ] Metadata store initialized     database_url=sqlite:///./data/graphrag.db
✓ Imports successful


In [ ]:
# Set your OpenAI API key here (or use environment variable)
# IMPORTANT: Never commit your actual API key to version control!
import os
# Option 1: Set it directly (only for local testing - remove before committing!)
# os.environ['OPENAI_API_KEY'] = 'your-api-key-here'
# Option 2: Use an existing environment variable (recommended)
if not os.getenv('OPENAI_API_KEY'):
    print("⚠️  OPENAI_API_KEY not set. Please set it in this cell or as an environment variable.")

## 1. Ingest Documents


In [4]:
# Step 0: Fetch PubMed data (if not already done)
test_dir = "./test_data/pubmed"
Path(test_dir).mkdir(parents=True, exist_ok=True)

# Check if PubMed data already exists
existing_files = glob.glob(f"{test_dir}/pubmed_*.md")
if not existing_files:
    print("Fetching PubMed data...")
    queries = get_sample_medical_queries()[:2]  # Use first 2 queries for demo
    files = create_pubmed_test_documents(queries, output_dir=test_dir)
    print(f"Created {len(files)} PubMed documents")
else:
    print(f"Using existing PubMed data: {len(existing_files)} files")
    files = existing_files

# Step 1: Ingest documents
doc_ids = await pipeline.ingest(
    paths=files,
    tags=["pubmed", "demo"]
)
print(f"Ingested {len(doc_ids)} documents")


Using existing PubMed data: 2 files
{"path": "./test_data/pubmed\\pubmed_asthma_treatment_0.md", "doc_id": "doc_3e47aa7bb9c5", "event": "Ingested file", "level": "info"}
{"path": "./test_data/pubmed\\pubmed_diabetes_management_1.md", "doc_id": "doc_b82b8a329ce6", "event": "Ingested file", "level": "info"}
{"doc_id": "doc_3e47aa7bb9c5", "event": "Document added", "level": "info"}
Load pretrained SentenceTransformer: sentence-transformers/all-mpnet-base-v2
Use pytorch device_name: cpu


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Load pretrained SentenceTransformer: sentence-transformers/all-mpnet-base-v2
Use pytorch device_name: cpu
Loading faiss with AVX2 support.
Successfully loaded faiss with AVX2 support.
{"path": "data\\faiss_index", "vectors": 143, "event": "Loaded FAISS index", "level": "info"}
{"path": "data\\faiss_index", "event": "Saved FAISS index", "level": "info"}
{"path": "data\\faiss_index", "event": "Saved FAISS index", "level": "info"}
{"path": "data\\faiss_index", "event": "Saved FAISS index", "level": "info"}
{"path": "data\\faiss_index", "event": "Saved FAISS index", "level": "info"}
{"path": "data\\faiss_index", "event": "Saved FAISS index", "level": "info"}
{"doc_id": "doc_3e47aa7bb9c5", "chunks": 5, "event": "Ingested document", "level": "info"}
{"doc_id": "doc_b82b8a329ce6", "event": "Document added", "level": "info"}


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

{"path": "data\\faiss_index", "event": "Saved FAISS index", "level": "info"}
{"path": "data\\faiss_index", "event": "Saved FAISS index", "level": "info"}
{"path": "data\\faiss_index", "event": "Saved FAISS index", "level": "info"}
{"path": "data\\faiss_index", "event": "Saved FAISS index", "level": "info"}
{"path": "data\\faiss_index", "event": "Saved FAISS index", "level": "info"}
{"path": "data\\faiss_index", "event": "Saved FAISS index", "level": "info"}
{"doc_id": "doc_b82b8a329ce6", "chunks": 6, "event": "Ingested document", "level": "info"}
Ingested 2 documents


## 2. Build Knowledge Graph


In [5]:
# Extract entities and relations, build graph
result = await pipeline.rebuild_graph(min_conf=0.4, batch_size=64)
print(f"Graph built: {result['entities']} entities, {result['relations']} relations")


HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
Load pretrained SentenceTransformer: sentence-transformers/all-mpnet-base-v2
Use pytorch device_name: cpu


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

{"input_count": 14, "output_groups": 14, "event": "Canonicalized entities", "level": "info"}
{"nodes": 147, "edges": 94, "event": "Loaded graph from database", "level": "info"}
Load pretrained SentenceTransformer: sentence-transformers/all-mpnet-base-v2
Use pytorch device_name: cpu
{"event": "NetworkX+Postgres graph store initialized", "level": "info"}


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

{"entities": 1751, "relations": 476, "event": "Graph rebuild completed", "level": "info"}
Graph built: 1751 entities, 476 relations


## 3. Query


In [9]:
# Query with automatic routing - using medical questions from PubMed data
request = QueryRequest(
    query="How does hypertension affect cardiovascular health?",
    mode="auto",  # or "graph" or "rag"
    k=10,
    hops=2,
    rerank=True
)

response = await query(request)
print(f"Route: {response.route}")
print(f"\nAnswer:\n{response.answer}")
print(f"\nCitations: {len(response.citations)}")
if response.citations:
    print("\nSample citations:")
    for citation in response.citations[:3]:
        print(f"  - {citation.doc_id}#{citation.chunk_id}: {citation.text[:100]}...")


HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

{"graph_count": 0, "vector_count": 20, "merged_count": 20, "event": "Aggregated results", "level": "info"}


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

{"query": "How does hypertension affect cardiovascular health", "input_count": 20, "output_count": 10, "event": "Reranked documents", "level": "info"}
HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
{"trace_id": "afb7c183-f9ac-49c4-a9a4-bd77800294d6", "event": "Trace saved", "level": "info"}
Route: rag

Answer:
Hypertension, or high blood pressure, can have detrimental effects on cardiovascular health. It can lead to conditions such as coronary artery disease, heart failure, stroke, and peripheral artery disease by putting strain on the heart and blood vessels, increasing the risk of damage and dysfunction [S:doc_2cf1a98f0673#doc_2cf1a98f0673_chunk_0]. Additionally, hypertension is a major risk factor for atherosclerosis, a condition where the arteries become narrowed and hardened due to a buildup of plaque, further contributing to cardiovascular problems [S:doc_5c1912b4b7cc#doc_5c1912b4b7cc_chunk_0]. Proper management of hypertension through lifestyle m